# Installing and Importing Libraries


In [0]:
%pip install databricks-feature-engineering
%pip install xgboost


In [0]:
dbutils.library.restartPython()


In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, FunctionTransformer, MinMaxScaler
import mlflow
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import ks_2samp
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression


# Sampling


This step aims to **build the dataset used for model training and evaluation**, integrating the information available in the Feature Store and performing a temporal split of the data.

First, the SQL query responsible for selecting records is loaded and executed. Next, Feature Store tables are used to retrieve the different categories of variables used in the model, such as registration, temporal, financial history, income, employee, and payment history information.

The FeatureEngineeringClient performs this integration through FeatureLookups, using ID_CLIENTE, ID_DOCUMENTO, and DATA_REF as lookup keys.

After creating the training set, the data is sorted by **reference date (DATA_REF)** and split temporally:

| Set        | Proportion |
| ---------- | ---------: |
| Train      |        60% |
| Validation |        20% |
| Test       |        20% |

The temporal split is used to **simulate a real credit scenario**, in which the model is trained using past information and later applied to future periods.

Finally, the FL_INAD variable is separated as the **target**, while the remaining variables are used as features:

* X_train, X_val, X_test: predictor variables;
* y_train, y_val, y_test: default target variable.

This approach avoids a random split of the data and reduces the risk of **temporal leakage**, providing an evaluation closer to the conditions of model usage in production.


In [ ]:
# Function to import an SQL query from a file
def import_query(path):
    with open(path) as f:
        return f.read()

# Load SQL query
query = import_query("fl_inad.sql")
df = spark.sql(query)

# List of FeatureLookups to retrieve features from the Feature Store (now in English and using English table names)
feature_lookups = [
    FeatureLookup(table_name="feature_store.credit_score.fs_cadastral", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_temporal", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_income_history", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_income", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_employees", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"]),
    FeatureLookup(table_name="feature_store.credit_score.fs_payment_history", lookup_key=["CLIENT_ID", "DOCUMENT_ID", "REF_DATE"])
]

# Create FeatureEngineeringClient object
fe = FeatureEngineeringClient()

# Create training_set with features and label
training_set = fe.create_training_set(df=df, feature_lookups=feature_lookups, label="DEFAULT_FLAG")

# Display the training_set DataFrame
training_set.load_df().display()

In [0]:
df_train  = training_set.load_df().toPandas()


In [0]:
# Sort the dataframe by reference time
df_sorted = df_train.sort_values('REF_DATE').reset_index(drop=True)

# Define train, validation, and test set sizes
n = len(df_sorted)
n_train = int(0.6 * n)
n_val   = int(0.2 * n)
n_test  = n - n_train - n_val

# Temporal split: first 60% for train, 20% for validation, 20% for test
train_df = df_sorted.iloc[:n_train]
val_df   = df_sorted.iloc[n_train:n_train + n_val]
test_df  = df_sorted.iloc[n_train + n_val:]

# Separate features and target
X_train, y_train = train_df.drop(columns=['FL_INAD'], errors='ignore'), train_df['FL_INAD']
X_val,   y_val   = val_df.drop(columns=['FL_INAD'], errors='ignore'),   val_df['FL_INAD']
X_test,  y_test  = test_df.drop(columns=['FL_INAD'], errors='ignore'),  test_df['FL_INAD']


# Exploration


In this step, an initial analysis of the training data is performed to **understand variable behavior, identify missing values, and define treatment strategies**.

First, variable means are compared between non-default and default customers, allowing relevant differences between groups to be identified. The percentage of null values for each variable is also calculated.

Next, a NullImputer is created to handle missing values. Imputation is done according to the meaning of each variable, using strategies such as **mode, mean by company size, median, constant values, and cascade filling**. Flags are also created indicating when certain values were originally missing, preserving that information for the model.

Finally, transformations are applied to variables, such as **date feature extraction, ordinal transformation of PORTE, and encoding of ESTADO and DDD**. After these steps, numeric variables to be used in the rest of preprocessing and modeling are identified automatically.


In [0]:
df_train_explore = X_train.copy()
df_train_explore["target"] = y_train

describe = (
    df_train_explore
    .groupby("target")
    .mean(numeric_only=True)
    .T
)

describe["variable"] = describe.index
describe["ratio"] = describe[1] / describe[0].replace(0, np.nan)

cols = ["variable"] + [col for col in describe.columns if col != "variable"]
describe = describe[cols]

display(describe)


In [0]:
# Calculate the percentage of NaN values per column
nan_pct = X_train.isna().mean() * 100

# Filter only columns with NaN and sort in descending order
nan_pct = nan_pct[nan_pct > 0].sort_values(ascending=False)

# Convert to DataFrame for visualization
nan_pct_df = nan_pct.to_frame('NaN_Percentage').reset_index().rename(columns={'index': 'Column'})

display(nan_pct_df)


In [0]:
# Null value imputer
class NullImputer(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass

    def fit(self, X, y=None):
        X = X.copy()
        X["SIZE"] = X["SIZE"].fillna("UNKNOWN")
        X["AREA_CODE"] = pd.to_numeric(X["AREA_CODE"], errors="coerce")

        # Modes for category imputation
        self.state_mode_ = X["STATE"].mode()[0]
        self.zip2_mode_ = X["ZIP_2_DIG"].mode()[0]
        self.region_mode_ = X["REGION"].mode()[0]
        self.area_code_mode_ = X["AREA_CODE"].mode()[0]
        self.area_code_by_state_ = (
            X.groupby("STATE")["AREA_CODE"]
            .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
            .to_dict()
        )

        # Imputation by company size mean and global mean for level variables
        self.level_cols = [
            'AVG_INCOME_3M', 'MIN_INCOME_3M', 'MAX_INCOME_3M', 'SUM_INCOME_3M',
            'AVG_INCOME_6M', 'MIN_INCOME_6M', 'MAX_INCOME_6M', 'SUM_INCOME_6M',
            'AVG_INCOME_1Y', 'MIN_INCOME_1Y', 'MAX_INCOME_1Y', 'SUM_INCOME_1Y',
            'AVG_INCOME_LIFETIME', 'MIN_INCOME_LIFETIME', 'MAX_INCOME_LIFETIME',
        ]
        self.size_means_ = {}
        self.global_means_ = {}
        for col in self.level_cols:
            self.size_means_[col] = X.groupby("SIZE")[col].mean().to_dict()
            self.global_means_[col] = X[col].mean()

        # Constants for imputation
        self.fill_zero_ = [
            'AVG_ADVANCE_DAYS_3M',
            'AVG_ADVANCE_DAYS_6M',
            'AVG_ADVANCE_DAYS_12M',
            'AVG_ADVANCE_DAYS_LIFETIME',
            'GROWTH_EMPLOYEES_3M',
            'GROWTH_EMPLOYEES_6M',
            'GROWTH_EMPLOYEES_12M',
            'GROWTH_EMPLOYEES_LIFETIME',
            'FLAG_SIZE_MISSING',
            'FLAG_HISTORY_3M',
            'FLAG_HISTORY_6M',
            'FLAG_HISTORY_12M',
            'NUM_HISTORY_HARVESTS',
            'FLAG_HISTORY_VARIATION',
            'MAX_MONTHLY_GROWTH',
            'MAX_MONTHLY_DROP',
        ]
        self.fill_9999_ = [
            'DAYS_SINCE_LAST_DELAY',
            'DAYS_SINCE_LAST_DEFAULT',
            'DAYS_LAST_PAYMENT',
        ]
        self.fill_minus1_ = [
            'NUM_EMPLOYEES_CURRENT',
            'NUM_EMPLOYEES_3M',
            'NUM_EMPLOYEES_6M',
            'NUM_EMPLOYEES_12M',
            'NUM_EMPLOYEES_LIFETIME',
        ]

        # Variation and std variables
        self.variation_cols = [
            'ABS_INCOME_GROWTH_3M', 'PERC_INCOME_GROWTH_3M',
            'ABS_INCOME_GROWTH_6M', 'PERC_INCOME_GROWTH_6M',
            'ABS_INCOME_GROWTH_1Y', 'PERC_INCOME_GROWTH_1Y',
            'MIN_INCOME_DIFF_3M', 'AVG_INCOME_DIFF_3M', 'MAX_INCOME_DIFF_3M',
            'MIN_INCOME_RATIO_3M', 'AVG_INCOME_RATIO_3M', 'MAX_INCOME_RATIO_3M',
            'MIN_INCOME_DIFF_6M', 'AVG_INCOME_DIFF_6M', 'MAX_INCOME_DIFF_6M',
            'MIN_INCOME_RATIO_6M', 'AVG_INCOME_RATIO_6M', 'MAX_INCOME_RATIO_6M',
            'MIN_INCOME_DIFF_1Y', 'AVG_INCOME_DIFF_1Y', 'MAX_INCOME_DIFF_1Y',
            'MIN_INCOME_RATIO_1Y', 'AVG_INCOME_RATIO_1Y', 'MAX_INCOME_RATIO_1Y',
            'MIN_INCOME_DIFF_LIFETIME', 'AVG_INCOME_DIFF_LIFETIME', 'MAX_INCOME_DIFF_LIFETIME',
            'MIN_INCOME_RATIO_LIFETIME', 'AVG_INCOME_RATIO_LIFETIME', 'MAX_INCOME_RATIO_LIFETIME',
        ]
        self.std_cols = [
            'STD_INCOME_3M', 'STD_INCOME_6M', 'STD_INCOME_1Y', 'STD_INCOME_LIFETIME',
        ]

        # Cascades for sequential imputation
        self.cascades = {
            'AVG_DELAY_DAYS':  ['AVG_DELAY_DAYS_3M', 'AVG_DELAY_DAYS_6M', 'AVG_DELAY_DAYS_12M'],
            'MIN_DELAY_DAYS':    ['MIN_DELAY_DAYS_3M', 'MIN_DELAY_DAYS_6M', 'MIN_DELAY_DAYS_12M'],
            'MAX_DELAY_DAYS':    ['MAX_DELAY_DAYS_3M', 'MAX_DELAY_DAYS_6M', 'MAX_DELAY_DAYS_12M'],
            'AVG_EMISSION_PAYMENT_DAYS': [
                'AVG_EMISSION_PAYMENT_DAYS_3M',
                'AVG_EMISSION_PAYMENT_DAYS_6M',
                'AVG_EMISSION_PAYMENT_DAYS_12M'
            ],
            'AVG_CHARGE_AMOUNT_PER_DAY': [
                'AVG_CHARGE_AMOUNT_PER_DAY_3M',
                'AVG_CHARGE_AMOUNT_PER_DAY_6M',
                'AVG_CHARGE_AMOUNT_PER_DAY_12M',
                'AVG_CHARGE_AMOUNT_PER_DAY_LIFETIME'
            ],
            'MIN_AMOUNT_TO_PAY': [
                'MIN_AMOUNT_TO_PAY_3M',
                'MIN_AMOUNT_TO_PAY_6M',
                'MIN_AMOUNT_TO_PAY_12M',
                'MIN_AMOUNT_TO_PAY_LIFETIME'
            ],
            'AVG_AMOUNT_TO_PAY': [
                'AVG_AMOUNT_TO_PAY_3M',
                'AVG_AMOUNT_TO_PAY_6M',
                'AVG_AMOUNT_TO_PAY_12M',
                'AVG_AMOUNT_TO_PAY_LIFETIME'
            ],
            'MAX_AMOUNT_TO_PAY': [
                'MAX_AMOUNT_TO_PAY_3M',
                'MAX_AMOUNT_TO_PAY_6M',
                'MAX_AMOUNT_TO_PAY_12M',
                'MAX_AMOUNT_TO_PAY_LIFETIME'
            ],
            'MIN_NUM_DELAYS': ['MIN_NUM_DELAYS_3M', 'MIN_NUM_DELAYS_6M', 'MIN_NUM_DELAYS_12M'],
            'AVG_NUM_DELAYS': ['AVG_NUM_DELAYS_3M', 'AVG_NUM_DELAYS_6M', 'AVG_NUM_DELAYS_12M'],
            'MAX_NUM_DELAYS': ['MAX_NUM_DELAYS_3M', 'MAX_NUM_DELAYS_6M', 'MAX_NUM_DELAYS_12M'],
            'PERC_ON_TIME': ['PERC_ON_TIME_3M', 'PERC_ON_TIME_6M', 'PERC_ON_TIME_12M'],
            'PERC_NOT_DEFAULT': ['PERC_NOT_DEFAULT_3M', 'PERC_NOT_DEFAULT_6M', 'PERC_NOT_DEFAULT_12M'],
            'AVG_ISSUE_TO_DUE_DAYS': [
                'AVG_ISSUE_TO_DUE_DAYS_3M',
                'AVG_ISSUE_TO_DUE_DAYS_6M',
                'AVG_ISSUE_TO_DUE_DAYS_12M'
            ],
            'MAX_ISSUE_TO_DUE_DAYS': [
                'MAX_ISSUE_TO_DUE_DAYS_3M',
                'MAX_ISSUE_TO_DUE_DAYS_6M',
                'MAX_ISSUE_TO_DUE_DAYS_12M'
            ],
            'MIN_ISSUE_TO_DUE_DAYS': [
                'MIN_ISSUE_TO_DUE_DAYS_3M',
                'MIN_ISSUE_TO_DUE_DAYS_6M',
                'MIN_ISSUE_TO_DUE_DAYS_12M'
            ],
        }
        self.cascade_medians_ = {}
        for _, cols in self.cascades.items():
            series = X[cols].bfill(axis=1).iloc[:,0]
            med = series.median()
            if pd.isna(med):
                med = 0
            self.cascade_medians_[cols[0]] = med

        # Medians for specific columns
        remaining_cols = [
            'AVG_CHARGE_AMOUNT_PER_DAY_6M',
            'AVG_AMOUNT_TO_PAY_LIFETIME',
            'MIN_AMOUNT_TO_PAY_LIFETIME',
            'MAX_AMOUNT_TO_PAY_LIFETIME',
            'MAX_AMOUNT_TO_PAY_6M',
            'MIN_AMOUNT_TO_PAY_6M',
            'AVG_AMOUNT_TO_PAY_6M',
            'AVG_EMISSION_PAYMENT_DAYS_6M',
            'AVG_ISSUE_TO_DUE_DAYS_6M',
            'MIN_ISSUE_TO_DUE_DAYS_6M',
            'PERC_ON_TIME_6M',
            'MAX_ISSUE_TO_DUE_DAYS_6M',
            'PERC_NOT_DEFAULT_6M',
            'MAX_DELAY_DAYS_6M',
            'MIN_DELAY_DAYS_6M',
            'AVG_DELAY_DAYS_6M',
            'MAX_NUM_DELAYS_6M',
            'MIN_NUM_DELAYS_6M',
            'AVG_NUM_DELAYS_6M',
            'AVG_CHARGE_AMOUNT_PER_DAY_12M',
            'MAX_ISSUE_TO_DUE_DAYS_12M',
            'MIN_ISSUE_TO_DUE_DAYS_12M',
            'AVG_EMISSION_PAYMENT_DAYS_12M',
            'AVG_ISSUE_TO_DUE_DAYS_12M',
            'AVG_AMOUNT_TO_PAY_12M',
            'PERC_NOT_DEFAULT_12M',
            'PERC_ON_TIME_12M',
            'MAX_AMOUNT_TO_PAY_12M',
            'MIN_AMOUNT_TO_PAY_12M',
            'MAX_DELAY_DAYS_12M',
            'MIN_DELAY_DAYS_12M',
            'AVG_DELAY_DAYS_12M',
            'MAX_NUM_DELAYS_12M',
            'MIN_NUM_DELAYS_12M',
            'AVG_NUM_DELAYS_12M',
            'AVG_CHARGE_AMOUNT_PER_DAY_LIFETIME'
        ]
        self.median_cols = remaining_cols + [
            'INCOME_PER_EMPLOYEE_RATIO',
            'PAYMENT_AMOUNT_RATE_RATIO',
            'DIFF_FROM_AVG_SIZE'
        ]
        self.medians_ = {}
        for col in self.median_cols:
            if col in X.columns:
                med = X[col].median()
                if pd.isna(med):
                    med = 0
                self.medians_[col] = med

        return self

    def transform(self, X):
        X = X.copy()
        X["AREA_CODE"] = pd.to_numeric(X["AREA_CODE"], errors="coerce")
        X["SIZE"] = X["SIZE"].fillna("UNKNOWN")
        X["INDUSTRIAL_SEGMENT"] = X["INDUSTRIAL_SEGMENT"].fillna("unknown")
        X["EMAIL_DOMAIN"] = X["EMAIL_DOMAIN"].fillna("UNKNOWN")
        X["AREA_CODE"] = X["AREA_CODE"].mask(
            X["AREA_CODE"].isna(),
            X["STATE"].map(self.area_code_by_state_)
        )
        X["AREA_CODE"] = X["AREA_CODE"].fillna(self.area_code_mode_)
        X["STATE"] = X["STATE"].fillna(self.state_mode_)
        X["ZIP_2_DIG"] = X["ZIP_2_DIG"].fillna(self.zip2_mode_)
        X["REGION"] = X["REGION"].fillna(self.region_mode_)

        # Fix FLAG_NO_EMPLOYEE_HISTORY
        X["FLAG_NO_EMPLOYEE_HISTORY"] = (
            X["FLAG_SIZE_MISSING"].isna().astype("int8")
        )

        # Imputation by constants
        for col in self.fill_zero_:
            if col in X.columns:
                X[col] = X[col].fillna(0)
        for col in self.fill_9999_:
            if col in X.columns:
                X[col] = X[col].fillna(9999)
        for col in self.fill_minus1_:
            if col in X.columns:
                X[col] = X[col].fillna(-1)

        # Imputation by company size mean and global mean
        for col in self.level_cols:
            X[f"FLAG_{col}_IMPUTED"] = X[col].isna().astype("int8")
            means = X["SIZE"].map(self.size_means_[col])
            X[col] = (
                X[col]
                .fillna(means)
                .fillna(self.global_means_[col])
            )

        # Flags and imputation for variation and std variables
        flag_variation = {}
        for col in self.variation_cols:
            if col in X.columns:
                flag_variation[f"FLAG_{col}_MISSING"] = X[col].isna().astype("int8")
                X[col] = X[col].fillna(0)
        flag_std = {}
        for col in self.std_cols:
            if col in X.columns:
                flag_std[f"FLAG_{col}_MISSING"] = X[col].isna().astype("int8")
                X[col] = X[col].fillna(0)
        if flag_variation or flag_std:
            X = pd.concat([X, pd.DataFrame({**flag_variation, **flag_std}, index=X.index)], axis=1)

        # Imputation for CONSECUTIVE_MONTHS_DROP
        if "CONSECUTIVE_MONTHS_DROP" in X.columns:
            X["FLAG_NO_INCOME_HISTORY"] = (
                X["CONSECUTIVE_MONTHS_DROP"]
                .isna()
                .astype("int8")
            )
            X["CONSECUTIVE_MONTHS_DROP"] = (
                X["CONSECUTIVE_MONTHS_DROP"]
                .fillna(0)
            )

        # Cascade imputation
        flag_cascade = {}
        for _, cols in self.cascades.items():
            main = cols[0]
            flag_cascade[f"FLAG_{main}_IMPUTED_CASCADE"] = (
                X[main].isna().astype("int8")
            )
            result = X[cols].bfill(axis=1).iloc[:,0]
            X[main] = result.fillna(
                self.cascade_medians_[main]
            )
        if flag_cascade:
            X = pd.concat([X, pd.DataFrame(flag_cascade, index=X.index)], axis=1)

        # Median imputation
        for col, med in self.medians_.items():
            if col in X.columns:
                X[col] = X[col].fillna(med)
        return X

# Date feature extraction
def add_date_features(X):
    X = X.copy()
    X['REF_YEAR'] = pd.to_datetime(X['REF_DATE']).dt.year
    X['REF_MONTH'] = pd.to_datetime(X['REF_DATE']).dt.month
    X['REGISTRATION_YEAR'] = pd.to_datetime(X['REGISTRATION_DATE']).dt.year
    X['REGISTRATION_MONTH'] = pd.to_datetime(X['REGISTRATION_DATE']).dt.month
    X = X.drop(columns=['REF_DATE', 'REGISTRATION_DATE'])
    return X

# Ordinal conversion for SIZE
def add_size_ordinal(X):
    X = X.copy()
    size_order = ['SMALL', 'MEDIUM', 'LARGE', 'UNKNOWN']
    X['SIZE'] = X['SIZE'].map({v: i for i, v in enumerate(size_order)})
    return X

# Custom label encoding for STATE and AREA_CODE
class LabelEncoderTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.state_map_ = None
        self.area_code_map_ = None

    def fit(self, X, y=None):
        X = X.copy()
        self.state_map_ = {
            v: i
            for i, v in enumerate(sorted(X["STATE"].astype(str).unique()))
        }
        self.area_code_map_ = {
            v: i
            for i, v in enumerate(sorted(X["AREA_CODE"].astype(str).unique()))
        }
        return self

    def transform(self, X):
        X = X.copy()
        X["STATE_LABEL"] = (
            X["STATE"]
            .astype(str)
            .map(self.state_map_)
            .fillna(-1)
            .astype(int)
        )
        X["AREA_CODE_LABEL"] = (
            X["AREA_CODE"]
            .astype(str)
            .map(self.area_code_map_)
            .fillna(-1)
            .astype(int)
        )
        X = X.drop(columns=["STATE", "AREA_CODE"])
        return X

# Preprocessing to define numeric columns
tmp = NullImputer().fit_transform(X_train)
tmp = add_date_features(tmp)
tmp = add_size_ordinal(tmp)
tmp = LabelEncoderTransformer().fit_transform(tmp)
num_cols = tmp.select_dtypes(include=np.number).columns.tolist()
del tmp


# Preprocessing


In this step, the **preprocessing pipeline** is defined, bringing together all required transformations in a single sequence.

The pipeline performs **missing value treatment**, **date variable extraction**, **PORTE transformation**, and **ESTADO and DDD encoding**. Next, categorical variables are transformed with **One-Hot Encoding**, while numeric variables are **normalized with MinMaxScaler**.

This way, all transformations are applied consistently to train, validation, and test data, avoiding preprocessing differences between sets.


In [0]:
# Preprocessing pipeline
preprocess_pipeline = Pipeline([
    ("null_imputer", NullImputer()),
    ("date_features", FunctionTransformer(add_date_features, validate=False)),
    ("size_ordinal", FunctionTransformer(add_size_ordinal, validate=False)),
    ("label_encodings", LabelEncoderTransformer()),
    ("column_transform", ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), ["INDUSTRY_SEGMENT", "REGION", "EMAIL_DOMAIN"]),
            ("num", MinMaxScaler(), num_cols)
        ],
        remainder="passthrough"
    ))
])


# Modeling


Three classification algorithms were tested: **XGBoost, Logistic Regression, and Random Forest**. For each model, the same preprocessing pipeline was used, ensuring a fair comparison.

Models were trained on the training set and evaluated on **train, validation, and test** sets. As main metrics, **AUC-ROC and KS** were used, allowing evaluation of both the ability to separate non-default and default customers and model stability across different sets.

In addition, the results of each experiment were logged in **MLflow**, making model comparison and tracking easier.

**Models evaluated:**

* **XGBoost:** tree-based model with boosting.
* **Logistic Regression:** linear model used as a reference and for greater interpretability.
* **Random Forest:** ensemble of decision trees capable of capturing nonlinear relationships.

At the end, the model with the best performance and greatest generalization capability was selected for the next steps.


## XGBoost


In [0]:
# Set MLflow experiment
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: preprocessing + XGBoost model
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("xgb_model", XGBClassifier(eval_metric="logloss"))
    ])

    # Train pipeline
    final_pipeline.fit(X_train, y_train)

    # Predictions for AUC calculation
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calculate AUC for each set
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Calculate KS for each set
    ks_train = ks_2samp(y_train_pred[y_train == 1], y_train_pred[y_train == 0]).statistic
    ks_val   = ks_2samp(y_val_pred[y_val == 1], y_val_pred[y_val == 0]).statistic
    ks_test  = ks_2samp(y_test_pred[y_test == 1], y_test_pred[y_test == 0]).statistic

    # Log metrics to MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)
    mlflow.log_metric("ks_train", ks_train)
    mlflow.log_metric("ks_val", ks_val)
    mlflow.log_metric("ks_test", ks_test)

    print(f"AUC train: {auc_train:.4f}, val: {auc_val:.4f}, test: {auc_test:.4f}")
    print(f"KS train: {ks_train:.4f}, val: {ks_val:.4f}, test: {ks_test:.4f}")


## Logistic Regression


In [0]:
# Set MLflow experiment
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: preprocessing + Logistic Regression model
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("logreg_model", LogisticRegression(max_iter=1000))
    ])

    # Train pipeline
    final_pipeline.fit(X_train, y_train)

    # Predictions for AUC calculation
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calculate AUC for each set
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Calculate KS for each set
    ks_train = ks_2samp(y_train_pred[y_train == 1], y_train_pred[y_train == 0]).statistic
    ks_val   = ks_2samp(y_val_pred[y_val == 1], y_val_pred[y_val == 0]).statistic
    ks_test  = ks_2samp(y_test_pred[y_test == 1], y_test_pred[y_test == 0]).statistic

    # Log metrics to MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)
    mlflow.log_metric("ks_train", ks_train)
    mlflow.log_metric("ks_val", ks_val)
    mlflow.log_metric("ks_test", ks_test)

    print(f"AUC train: {auc_train:.4f}, val: {auc_val:.4f}, test: {auc_test:.4f}")
    print(f"KS train: {ks_train:.4f}, val: {ks_val:.4f}, test: {ks_test:.4f}")


## Random Forest


In [0]:
# Set MLflow experiment
mlflow.set_experiment(experiment_id=3284700244160249)

with mlflow.start_run() as run:

    # Pipeline: preprocessing + Random Forest model
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("rf_model", RandomForestClassifier(n_estimators=100, random_state=42))
    ])

    # Train pipeline
    final_pipeline.fit(X_train, y_train)

    # Predictions for AUC calculation
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calculate AUC for each set
    auc_train = roc_auc_score(y_train, y_train_pred)
    auc_val   = roc_auc_score(y_val, y_val_pred)
    auc_test  = roc_auc_score(y_test, y_test_pred)

    # Calculate KS for each set
    ks_train = ks_2samp(y_train_pred[y_train == 1], y_train_pred[y_train == 0]).statistic
    ks_val   = ks_2samp(y_val_pred[y_val == 1], y_val_pred[y_val == 0]).statistic
    ks_test  = ks_2samp(y_test_pred[y_test == 1], y_test_pred[y_test == 0]).statistic

    # Log metrics to MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)
    mlflow.log_metric("ks_train", ks_train)
    mlflow.log_metric("ks_val", ks_val)
    mlflow.log_metric("ks_test", ks_test)

    print(f"AUC train: {auc_train:.4f}, val: {auc_val:.4f}, test: {auc_test:.4f}")
    print(f"KS train: {ks_train:.4f}, val: {ks_val:.4f}, test: {ks_test:.4f}")


# Final Model



After comparing the models, **XGBoost** was selected for the final model. The complete pipeline, including preprocessing and the model, was trained again and registered in **MLflow**.

The performance obtained was:

| Metric | Train  | Validation | Test   |
| ------ | -----: | ---------: | -----: |
| **AUC** | 0.9964 |     0.9759 | 0.9748 |
| **KS**  | 0.9437 |     0.8471 | 0.8497 |

Validation and test results are quite close, indicating **good generalization capability**. The model achieved AUC of **0.9748** and KS of **0.8497** on the test set.

Finally, the complete pipeline was **saved to MLflow**, allowing versioning and later use for new predictions.


## Final XGBoost


In [0]:
# Start MLflow run
with mlflow.start_run() as run:

    # Pipeline: preprocessing + XGBoost model
    final_pipeline = Pipeline([
        ("preprocessing", preprocess_pipeline),
        ("xgb_model", XGBClassifier(
            eval_metric="logloss" 
        ))
    ])

    # Train pipeline
    final_pipeline.fit(X_train, y_train)

    # Predictions for AUC calculation
    y_train_pred = final_pipeline.predict_proba(X_train)[:, 1]
    y_val_pred   = final_pipeline.predict_proba(X_val)[:, 1]
    y_test_pred  = final_pipeline.predict_proba(X_test)[:, 1]

    # Calculate KS for each set
    ks_train = ks_2samp(y_train_pred[y_train == 1], y_train_pred[y_train == 0]).statistic
    ks_val   = ks_2samp(y_val_pred[y_val == 1], y_val_pred[y_val == 0]).statistic
    ks_test  = ks_2samp(y_test_pred[y_test == 1], y_test_pred[y_test == 0]).statistic

    # Log metrics to MLflow
    mlflow.log_metric("auc_train", auc_train)
    mlflow.log_metric("auc_val", auc_val)
    mlflow.log_metric("auc_test", auc_test)
    mlflow.log_metric("ks_train", ks_train)
    mlflow.log_metric("ks_val", ks_val)
    mlflow.log_metric("ks_test", ks_test)
    
    # Save model to MLflow
    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        name="model",
        input_example=X_train.iloc[:5]
    )

    print(run.info.run_id)  # Display MLflow run_id


# Evaluation


## Loading the Model and Predicting on Validation and Test


In [0]:
model = mlflow.sklearn.load_model("runs:/e0d38d0f7d3a4d2a957ef79069735096/model")


In [0]:
# Create test and validation DataFrames with target
df_test = X_test.copy()
df_test["target"] = y_test

df_val = X_val.copy()
df_val["target"] = y_val

# Generate model predictions and probabilities
df_val["pred"] = model.predict(df_val)
df_val["proba"] = model.predict_proba(df_val)[:, 1]
df_test["pred"] = model.predict(df_test)
df_test["proba"] = model.predict_proba(df_test)[:, 1]


## Default Rate by Model Score


This analysis was performed to **verify whether the score generated by the model truly represents different levels of credit risk**. For this, customers are split into probability bands and the observed default rate is calculated for each band.

First, the probabilities generated by the model are divided into risk intervals:

| Score band  | Interpretation |
| ----------- | -------------- |
| 0.00 – 0.10 | Low risk       |
| 0.10 – 0.20 | Risk           |
| 0.20 – 0.30 | Risk           |
| 0.30 – 0.40 | Risk           |
| 0.40 – 0.50 | Risk           |
| 0.50 – 0.60 | High risk      |
| 0.60 – 1.00 | High risk      |

Then, for each band, the following are counted:

* **Total customers:** number of customers in that band;
* **Defaults:** number of default customers;
* **Default rate:** proportion of defaults within the band.

The same analysis is performed separately on the **validation and test** sets. This makes it possible to verify whether the behavior observed in training/validation is maintained on data the model did not use for fitting.

Next, default rates from the validation and test sets are grouped and an **average between the two sets** is calculated, providing an overall view of model behavior.

Finally, a bar chart is generated comparing the default rate of each band between validation and test.

The main goal is to verify whether there is an **increasing relationship between the model score and observed default**. A well-calibrated model, or at least one that is well ordered in terms of risk, should generally show **lower default rates in lower-score bands and higher rates in higher-score bands**.


In [0]:
# Define risk bands for the model score
bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0]
labels = [f"{bins[i]:.2f}-{bins[i+1]:.2f}" for i in range(len(bins)-1)]

df_val['risk_band'] = pd.cut(df_val['proba'], bins=bins, labels=labels, include_lowest=True, right=False)
df_test['risk_band'] = pd.cut(df_test['proba'], bins=bins, labels=labels, include_lowest=True, right=False)


In [0]:
# Group by risk_band and calculate default percentage for validation
df_val_grouped = df_val.groupby('risk_band', observed=False).agg(
    total_val=('target', 'count'),
    defaults_val=('target', 'sum')
)
df_val_grouped['default_pct_val'] = 100 * df_val_grouped['defaults_val'] / df_val_grouped['total_val']

# Group by risk_band and calculate default percentage for test
df_test_grouped = df_test.groupby('risk_band', observed=False).agg(
    total_test=('target', 'count'),
    defaults_test=('target', 'sum')
)
df_test_grouped['default_pct_test'] = 100 * df_test_grouped['defaults_test'] / df_test_grouped['total_test']

# Join default percentages from validation and test
df_grouped = df_val_grouped[['default_pct_val']].join(
    df_test_grouped[['default_pct_test']],
    how='outer'
)

# Calculate the average default percentages
df_grouped['avg_default_pct'] = df_grouped[
    ['default_pct_val', 'default_pct_test']
].mean(axis=1)

# Format the average as an integer percentage
df_grouped['avg_default_pct'] = df_grouped['avg_default_pct'].apply(lambda x: f"{x:.0f}%")

# Keep only risk_band and average for visualization
df_grouped = df_grouped[['avg_default_pct']].reset_index()

display(df_grouped)


In [0]:
import plotly.graph_objects as go

# Join default percentages from validation and test
df_hist = df_val_grouped[['default_pct_val']].join(
    df_test_grouped[['default_pct_test']],
    how='outer'
)

labels = df_hist.index.astype(str)

fig = go.Figure(data=[
    go.Bar(
        name='Validation',
        x=labels,
        y=df_hist['default_pct_val'],
        marker_color='rgb(55,55,55)',
        opacity=0.55
    ),
    go.Bar(
        name='Teste',
        x=labels,
        y=df_hist['default_pct_test'],
        marker_color='black',
        opacity=1.0
    )
])

fig.update_layout(
    barmode='group',
    xaxis_title='Risk Band',
    yaxis_title='Default Percentage',
    title='Default Percentage by Risk Band',
    legend_title=None,
    xaxis_tickangle=-45,
    template='simple_white',
    width=1000,
    height=600
)

fig.show()


# Financial Analysis


## Threshold Analysis


This step aims to **define the best cutoff point to convert the default probability generated by the model into a binary classification**.

Thresholds between **0 and 1** were tested, with intervals of 0.01, using the validation set. For each threshold, **F1 Score** was calculated, seeking a balance between precision and recall.

The threshold with the highest F1 Score was selected and later applied **without changes to the test set**, avoiding use of the test set in model selection.

On the test set, **accuracy, precision, recall, F1 Score, and AUC-ROC** were calculated, in addition to the confusion matrix, allowing evaluation of both overall performance and the model's ability to correctly identify default customers.


In [0]:
# Find the best threshold to maximize F1 score on the validation set
y_val_proba = df_val['proba']
best_threshold = 0.5
best_f1 = 0

thresholds = np.arange(0.0, 1.01, 0.01)
f1_scores = []

for threshold in thresholds:
    y_pred = (y_val_proba >= threshold).astype(int)
    score = f1_score(y_val, y_pred)
    f1_scores.append(score)
    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print(f"Best threshold for F1 on the validation set: {best_threshold:.2f} (F1={best_f1:.4f})")

# Plot F1 vs Threshold curve
plt.figure(figsize=(8,5))
plt.plot(thresholds, f1_scores, marker='o')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score by Threshold on the validation set')
plt.grid(True)
plt.show()


In [0]:
# Generate binary predictions with threshold 0.34 for the test set
y_test_proba = df_test['proba']
threshold = 0.34
y_test_pred_t34 = (y_test_proba >= threshold).astype(int)

# Confusion matrix for threshold 0.34
cm = confusion_matrix(y_test, y_test_pred_t34)
print("Confusion matrix for threshold 0.34 on the test set:")
print(cm)

# Calculate evaluation metrics
acc = accuracy_score(y_test, y_test_pred_t34)
prec = precision_score(y_test, y_test_pred_t34)
rec = recall_score(y_test, y_test_pred_t34)
f1 = f1_score(y_test, y_test_pred_t34)
auc = roc_auc_score(y_test, y_test_proba)  # AUC uses the probabilities

print(f"\nAccuracy:  {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC:       {auc:.4f}")


## Credit Policy vs Model


This step aims to **compare the financial impact of the credit model with a credit policy used as a proxy**.

First, the amount due for each document is retrieved from the database and associated with the test set. Next, credit decisions are made using two approaches: the **proxy**, based on the customer's default history, and the **model**, based on default probability.

To make the comparison fair, the model is adjusted to **deny the same percentage of documents as the proxy**. This way, both strategies operate with approximately the same approval rate, allowing evaluation of which one selects a higher-quality portfolio.

The following are compared:

* **Approval rate:** percentage of approved customers;
* **Default rate among approved:** percentage of default customers within the approved portfolio;
* **Loss Rate:** proportion of approved value that results in loss;
* **Approved value:** sum of values for approved documents;
* **Lost value:** value associated with approved default customers;
* **Financial result:** value from approved non-default customers minus value from approved default customers.

Finally, differences between the model and the proxy are calculated, making it possible to verify **how much more the model generates and how much it reduces financial losses**, while maintaining the same approval level as the reference policy.


In [0]:
# Fetch amount due per document from the database
valor_a_pagar_df = spark.sql("""
    SELECT ID_DOCUMENTO, VALOR_A_PAGAR AS PAYMENT_VALUE
    FROM credit_score.data.pagamentos
""").toPandas()

# Add VALOR_A_PAGAR column to df_test, matching column names
df_test = df_test.merge(valor_a_pagar_df, left_on="DOCUMENT_ID", right_on="ID_DOCUMENTO", how="left")


In [0]:
# Proxy prediction: approve if PCT_FORA_INADIMPLENCIA_12M < 0.8
df_test['proxy_pred'] = (df_test['PCT_FORA_INADIMPLENCIA_12M'] < 0.8).astype(int)
# Model prediction: approve if proba >= 0.25
df_test['pred'] = (df_test['proba'] >= 0.25).astype(int)

# Percentage of documents denied by the proxy
percent_denied_proxy = (df_test['proxy_pred'] == 1).mean()
print(f"Percentage of documents denied by the proxy: {percent_denied_proxy:.2%}")

# Total payment value gained by the proxy (approved non-default minus approved default)
proxy_approved_non_default_value = df_test.loc[(df_test['proxy_pred'] == 0) & (df_test['target'] == 0), 'PAYMENT_VALUE'].sum()
proxy_approved_default_value = df_test.loc[(df_test['proxy_pred'] == 0) & (df_test['target'] == 1), 'PAYMENT_VALUE'].sum()
proxy_total_value_gained = proxy_approved_non_default_value - proxy_approved_default_value
print(f"Proxy - Total value gained: {proxy_total_value_gained:.0f}")

# Payment value lost by the proxy (denied non-default)
proxy_value_lost = df_test.loc[(df_test['proxy_pred'] == 1) & (df_test['target'] == 0), 'PAYMENT_VALUE'].sum()
print(f"Proxy - Value lost: {proxy_value_lost:.0f}")

# Set model cutoff to deny the same percentage of rows as the proxy
model_threshold = df_test['proba'].quantile(1 - percent_denied_proxy)
df_test['pred_equal_proxy'] = (df_test['proba'] >= model_threshold).astype(int)

# Total payment value gained by the model (approved non-default minus approved default)
model_approved_non_default_value = df_test.loc[(df_test['pred_equal_proxy'] == 0) & (df_test['target'] == 0), 'PAYMENT_VALUE'].sum()
model_approved_default_value = df_test.loc[(df_test['pred_equal_proxy'] == 0) & (df_test['target'] == 1), 'PAYMENT_VALUE'].sum()
model_total_value_gained = model_approved_non_default_value - model_approved_default_value
print(f"Model - Total value gained: {model_total_value_gained:.0f}")

# Payment value lost by the model (denied non-default)
model_value_lost = df_test.loc[(df_test['pred_equal_proxy'] == 1) & (df_test['target'] == 0), 'PAYMENT_VALUE'].sum()
print(f"Model - Value lost: {model_value_lost:.0f}")

# Comparison: how much more the model generates than the proxy with equal cutoff
value_generated_additional = model_total_value_gained - proxy_total_value_gained
print(f"\nComparison: The model generates more than the proxy: {value_generated_additional:.0f}")

# Comparison: how much less the model loses compared to the proxy
loss_reduction = proxy_value_lost - model_value_lost
print(f"Comparison: The model has a loss reduction compared to the proxy of: {loss_reduction:.0f}")


In [0]:
# Calculate default rate among approvals by proxy and by model
default_rate_approved_proxy = df_test.loc[df_test['proxy_pred'] == 0, 'target'].mean()
default_rate_approved_model = df_test.loc[df_test['pred_equal_proxy'] == 0, 'target'].mean()

# Sum of values approved by proxy and model
approved_value_proxy = df_test.loc[df_test['proxy_pred'] == 0, 'PAYMENT_VALUE'].sum()
approved_value_model = df_test.loc[df_test['pred_equal_proxy'] == 0, 'PAYMENT_VALUE'].sum()

# Calculate loss: value lost among approved defaults
lost_value_approved_defaults_proxy = df_test.loc[(df_test['proxy_pred'] == 0) & (df_test['target'] == 1), 'PAYMENT_VALUE'].sum()
lost_value_approved_defaults_model_eq = df_test.loc[(df_test['pred_equal_proxy'] == 0) & (df_test['target'] == 1), 'PAYMENT_VALUE'].sum()

# Calculate loss rate (percentage lost over approved value)
loss_rate_proxy = lost_value_approved_defaults_proxy / approved_value_proxy if approved_value_proxy != 0 else float('nan')
loss_rate_model = lost_value_approved_defaults_model_eq / approved_value_model if approved_value_model != 0 else float('nan')

# Approval percentage by proxy and model
approval_rate_proxy = (df_test['proxy_pred'] == 0).mean()
approval_rate_model = (df_test['pred_equal_proxy'] == 0).mean()

# Extra value generated by model vs proxy, using correct variables from previous cell
value_generated_extra_eq = model_total_value_gained - proxy_total_value_gained
total_value_gained_proxy = proxy_total_value_gained

# Percentage of additional value generated by the model compared to proxy
percent_value_generated_extra_eq = (value_generated_extra_eq / abs(total_value_gained_proxy) * 100
                                    if total_value_gained_proxy != 0 else float('nan'))

# Value lost by proxy (denied non-default), and loss reduction (already computed before)
lost_value_proxy = proxy_value_lost
loss_reduction = proxy_value_lost - model_value_lost

# Percentage of value lost less by model compared to proxy
percent_loss_reduction = (loss_reduction / abs(lost_value_proxy) * 100
                          if lost_value_proxy != 0 else float('nan'))


In [0]:
print(f"Proxy approval: {approval_rate_proxy:.2%}")
print(f"Model approval: {approval_rate_model:.2%}\n")

print(f"Default rate among proxy-approved: {default_rate_approved_proxy:.2%}")
print(f"Default rate among model-approved: {default_rate_approved_model:.2%}\n")

print(f"Proxy loss rate: {loss_rate_proxy:.2%}")
print(f"Model loss rate: {loss_rate_model:.2%}\n")

print(f"Proxy approved value: {approved_value_proxy:.0f}")
print(f"Model approved value: {approved_value_model:.0f}\n")

print(f"Proxy lost value: {lost_value_approved_defaults_proxy:.0f}")
print(f"Model lost value: {lost_value_approved_defaults_model_eq:.0f}\n")

print(f"Percentage of additional value generated by the model: {percent_value_generated_extra_eq:.0f}%")
print(f"Percentage of model loss reduction: {percent_loss_reduction:.0f}%")


# Conclusion



The model showed **good ability to separate customers with different risk levels**. The probability bands show a clear relationship between the risk predicted by the model and observed default:

| Risk band   | Default rate |
| ----------- | -----------: |
| 0.00 – 0.10 |       **1%** |
| 0.10 – 0.20 |      **24%** |
| 0.20 – 0.30 |      **34%** |
| 0.30 – 0.40 |      **40%** |
| 0.40 – 0.50 |      **50%** |
| 0.50 – 0.60 |      **56%** |
| 0.60 – 1.00 |      **79%** |

This indicates that, as predicted probability increases, the default rate also increases, showing **good ordering of customers by risk**.

### Model performance

The best threshold found in validation was **0.34**, with F1 Score of **0.7024**.

| Metric   |       Test |
| -------- | ---------: |
| AUC-ROC  | **0.9748** |
| Accuracy | **95.74%** |
| Precision | **68.62%** |
| Recall   | **65.52%** |
| F1 Score | **0.6703** |

### Financial impact

To compare the model fairly with the credit policy used as a proxy, both were evaluated with the **same approval rate: 90.23%**.

| Indicator                   |       Proxy |          Model |
| --------------------------- | ----------: | -------------: |
| Approval                    |      90.23% |      **90.23%** |
| Default rate among approved |       2.64% |       **1.23%** |
| Loss Rate                   |       1.99% |       **1.20%** |
| Approved value              | R$ 1.346 bn | **R$ 1.359 bn** |
| Lost value                  |  R$ 67.1 m  |  **R$ 43.9 m** |
| Financial result            | R$ 1.293 bn | **R$ 1.327 bn** |

The model therefore showed:

* **53% less default among approved customers**;
* **34% reduction in financial losses**;
* approximately **R$ 23.1 million less in losses**;
* approximately **R$ 33.7 million in additional result**;
* **3% increase in generated value**.

### Final conclusion

Overall, the results show that the model can **maintain the same approval rate as the current policy while significantly reducing portfolio risk**. In addition to good statistical performance, the financial analysis indicates relevant improvement potential in credit decision-making, mainly through reduction of losses and default among approved customers.

> **In summary: the model can approve the same number of customers while selecting a portfolio with lower risk and lower financial loss.**


##
